# Lesson 4 — Target 변환, 오류 분석, 제출

**예상 시간:** 개념 40분 + 실습 90분  
**오늘의 새 산출물:** 선택된 target 전략, 오류 분석, 검증된 로컬 Kaggle 제출 파일

Lesson 3에서 validation으로 선택한 feature set을 고정한 뒤 시작합니다. 실제 과제의 완성 코드는 포함하지 않습니다.

## 1. 오늘의 질문

> 실제 수요 10대를 20대로 예측한 오류와 500대를 510대로 예측한 오류는 운영상 같은 의미일까? 최종 선택이 끝난 뒤 test 예측은 어떤 순서로 만들어야 할까?

오늘은 feature 실험을 끝내고 target 표현과 오류를 검증한 뒤 재현 가능한 제출 파일을 만듭니다.

## 2. 선수 지식 확인

- Lesson 3의 최종 feature 이름과 pipeline을 재현할 수 있는가?
- feature 선택에 공식 test를 쓰지 않았는가?
- MAE와 RMSE가 실제값과 예측값을 비교한다는 점을 설명할 수 있는가?

## 3. 개념 설명

### 3.1 왜 target을 변환하는가?

자전거 수요는 한산한 시간의 작은 값과 출퇴근 시간의 큰 값이 함께 있어 오른쪽 꼬리가 길 수 있습니다. 원래 `count`로 학습하면 큰 절대오차가 학습에 강한 영향을 줄 수 있습니다. `log1p(count)`는 큰 값 사이의 거리를 압축하면서 0도 안전하게 변환합니다.

`np.log1p(x)`는 `log(1+x)`의 새 값 또는 배열을 반환하고 원본을 바꾸지 않습니다. 예측을 원래 단위로 되돌릴 때 `np.expm1`을 사용합니다. 결과는 재할당해야 합니다.

### 3.2 RMSLE

RMSLE는 실제값과 예측값에 각각 `log1p`를 적용한 뒤 차이의 제곱평균 제곱근을 계산합니다. 절대 10대 차이보다 상대적인 크기 차이에 더 민감합니다. 음수에는 로그를 적용할 수 없으므로 예측을 0 이상으로 제한합니다.

log target 학습이 RMSLE를 반드시 개선하는 것은 아닙니다. 원 target 전략과 같은 OOF 행에서 비교해 선택합니다.

### 3.3 오류 분석

전체 점수 하나만으로는 모델이 언제 실패하는지 알 수 없습니다. `hour`, `workingday`, `weather`, 실제 수요 수준별로 오류를 묶어 봅니다. 큰 오류가 특정 시간에 모여도 그 시간 자체가 오류의 원인이라고 증명되지는 않습니다.

### 3.4 최종 재학습과 제출

feature set과 target 전략을 validation으로 확정한 뒤 전체 train을 사용해 pipeline을 새로 fit합니다. 동일한 feature 함수를 test에 적용하고 원래 test 순서를 유지해 `datetime,count`를 저장합니다. `to_csv(..., index=False)`는 파일을 쓰며 DataFrame 원본은 바꾸지 않습니다.

## 4. 손으로 만드는 작은 표

| 실제값 | 예측값 | 절대오차 | 직관적인 상대 차이 |
|---:|---:|---:|---|
| 10 | 20 | 10 | 약 2배 예측 |
| 500 | 510 | 10 | 약 2% 높게 예측 |

MAE에는 두 행의 절대오차가 모두 10이지만 RMSLE는 첫 번째 행을 더 큰 오류로 봅니다. 대회는 시간당 수요 규모가 크게 다른 상황에서 이 로그 척도를 사용합니다.

## 5. 실행 가능한 toy example

실제 과제와 다른 매장 주문값으로 변환, 복원, RMSLE, 제출 형식 검증을 연습합니다.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_log_error

actual = np.array([0, 10, 500], dtype=float)
prediction = np.array([-2, 20, 510], dtype=float)

safe_prediction = np.clip(prediction, 0, None)
logged = np.log1p(actual)
restored = np.expm1(logged)
rmsle = np.sqrt(mean_squared_log_error(actual, safe_prediction))
mae = mean_absolute_error(actual, safe_prediction)

print('log1p:', logged)
print('복원:', restored)
print('RMSLE:', rmsle)
print('MAE:', mae)

toy_submission = pd.DataFrame({
    'datetime': ['2024-01-01 00:00:00', '2024-01-01 01:00:00'],
    'count': np.clip([4.2, 8.7], 0, None),
})
print(toy_submission)
print('컬럼:', toy_submission.columns.tolist(), '행 수:', len(toy_submission))

`np.clip(prediction, 0, None)`은 0보다 작은 값을 0으로 바꾼 새 배열을 반환합니다. 원래 `prediction`은 바뀌지 않습니다. `mean_squared_log_error`는 MSL​​E이므로 마지막에 제곱근을 취해 RMSLE를 만듭니다.

대표 오류는 log target으로 학습한 예측을 `expm1`으로 복원하지 않고 실제 `count`와 비교하는 것입니다. 서로 다른 단위이므로 평가가 성립하지 않습니다.

## 6. Data Leakage 점검

- 원 target/log target 전략 선택에 공식 test를 사용하지 않았는가?
- 두 전략이 같은 feature set, fold, 모델 설정, validation 행을 사용하는가?
- log 예측을 원래 단위로 복원한 뒤 평가했는가?
- 최종 결정 전 전체 train으로 fit하지 않았는가?
- submission의 `datetime` 순서가 원본 test와 정확히 같은가?

## 7. 실제 데이터 Exercise

Lesson 3의 최종 feature set과 RandomForest 설정을 고정합니다. 원래 `count`를 학습한 전략과 `log1p(count)`를 학습하고 `expm1`으로 복원한 전략만 비교합니다. OOF RMSLE가 낮은 전략을 최종 선택한 뒤에만 전체 train 재학습과 test 예측을 수행합니다.

## 8. 코드 과제

**제출 경로:** `Sparta/competitions/bike-sharing-demand/answers/code/lesson4.ipynb`

- **C1 — Target 분포:** `count`의 최소·중앙값·평균·최대와 histogram을 출력한다. `log1p(count)` histogram도 같은 범위의 train에서 그린다.
- **C2 — 전략 비교:** Lesson 3 feature set으로 원 target과 log target 전략을 같은 3개 fold에서 비교한다. log 예측은 `expm1` 후 0 이상으로 제한한다.
- **C3 — 최종 선택:** fold별/OOF RMSLE와 MAE 표를 출력하고 OOF RMSLE가 낮은 target 전략을 코드 변수로 명시한다.
- **C4 — 오류 분석:** 선택 전략의 OOF 결과표에 `datetime`, actual, prediction, signed_error, absolute_error, `hour`, `workingday`, `weather`를 담는다. 시간·근무일·날씨별 MAE와 실제 수요 상위 10%의 MAE, 가장 큰 absolute error 10행을 출력한다.
- **C5 — 전체 재학습:** 최종 feature 함수와 pipeline을 전체 train에 fit하고 원본 순서를 유지한 test를 예측한다.
- **C6 — 제출 검증:** `Sparta/competitions/bike-sharing-demand/output/bike_submission.csv`를 `index=False`로 저장한다. 6,493행, `datetime,count` 컬럼, sample과 같은 datetime 순서, count의 결측·무한대·음수 0개를 assertion으로 검증한다.

**코드 최소 통과 기준**

- 두 target 전략 외의 feature/model/split 조건이 동일하다.
- OOF RMSLE로 최종 전략을 선택한다.
- 오류 분석이 validation 예측만 사용한다.
- train/test가 동일한 feature 함수와 학습된 pipeline을 사용한다.
- 모든 제출 assertion이 통과한다.

## 9. 글 과제

**제출 경로:** `Sparta/competitions/bike-sharing-demand/answers/text/lesson4.txt`

- **T1:** MAE가 같은 10→20과 500→510 예측을 RMSLE가 다르게 보는 이유를 설명한다. **통과 기준:** 상대적 크기 차이와 `log1p`를 연결한다.
- **T2:** 두 target 전략의 OOF RMSLE/MAE를 적고 최종 전략을 선택한다. **통과 기준:** 실제 수치를 사용하고 RMSLE를 주 선택 기준으로 삼는다.
- **T3:** 시간·근무일·날씨별 오류와 최대 오류 10행에서 관찰한 패턴을 설명한다. **통과 기준:** 관찰과 원인 추정을 구분하고 최소 두 구간의 수치를 사용한다.
- **T4:** validation 선택 후 전체 train으로 재학습하는 이유와 test를 그 전에 사용하면 안 되는 이유를 설명한다. **통과 기준:** 모델 선택과 최종 예측의 역할을 구분한다.
- **T5:** 제출 파일 검증 결과와 현실 운영상의 한계 한 가지를 적는다. **통과 기준:** 행 수·순서·값 범위 검증과 날씨 예보 가용성 또는 미래 성능 변화 중 하나를 포함한다.

## 10. 성찰 질문

**제출 불필요:** leaderboard 점수를 보지 않고 로컬 validation만으로 모델을 선택하는 과정이 왜 실제 수요예측 업무에 더 가까운 연습일까요?

## 11. 제출 전 자체 점검

- [ ] C1~C6, T1~T5가 모두 있는가?
- [ ] log 예측을 복원한 뒤 실제 count와 평가했는가?
- [ ] 오류 분석에서 인과관계를 단정하지 않았는가?
- [ ] 제출 파일 assertion이 모두 통과했는가?
- [ ] Kaggle 업로드 없이 로컬 파일까지만 만들었는가?